# 01 — LangGraph foundations: customer support workflow

We will build a small customer support workflow for a ticket about a duplicate charge. Starting with an `AgentState` and a straight-line **DAG**, we will add category routing, a bounded review loop, checkpoint persistence, conversation memory, human approval for a refund, and editing saved state.

The core examples are deterministic. An optional model-backed node at the end uses the API key and model name loaded from `.env`.

> Install the project dependencies first. The examples use `langgraph` and `langchain-core`. For the optional OpenAI model example, set `OPENAI_API_KEY` and `OPENAI_MODEL` in `.env`.

## 1. The graph mental model

Our running example is a support ticket: a customer says they were charged twice and wants a refund. We will model the workflow as small steps that share ticket data.

- **State** holds the ticket details and work completed so far.
- **Nodes** classify, draft, or review the response.
- **Edges** move the ticket through the workflow.

A simple workflow is a DAG (directed acyclic graph): every path moves forward and eventually ends. We will start with a basic ticket intake path, then add routing and a review cycle.

In [1]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph


load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL")

model = None
if OPENAI_API_KEY and OPENAI_MODEL:
    model = ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=OPENAI_API_KEY,
        reasoning_effort="none",
    )
    print(f"Loaded model configuration: {OPENAI_MODEL}")
else:
    print("Set OPENAI_API_KEY and OPENAI_MODEL in .env to enable the model example.")

Loaded model configuration: gpt-5.6-luna


### Define `AgentState`

`TypedDict` documents the ticket fields that nodes may read and update. Each node returns a partial dictionary; LangGraph merges those updates into the shared state.

In [2]:
class AgentState(TypedDict):
    ticket_id: str
    customer_message: str
    category: str
    summary: str
    draft_reply: str

### Build a straight-line DAG

The `classify_ticket` node identifies the billing issue, and `draft_reply` prepares a response. This first graph follows one path:

`START → classify_ticket → draft_reply → END`

In [3]:
def classify_ticket(state: AgentState) -> dict:
    """Classify the incoming support ticket."""
    message = state["customer_message"].lower()
    category = "billing" if "charged" in message else "general_support"
    summary = "Customer reports a duplicate charge and requests a refund."
    return {"category": category, "summary": summary}


def draft_reply(state: AgentState) -> dict:
    """Draft a response for the classified ticket."""
    reply = (
        f"Hi, I’m sorry about the duplicate charge on ticket "
        f"{state['ticket_id']}. We’ll review the transaction and follow up."
    )
    return {"draft_reply": reply}


builder = StateGraph(AgentState)
builder.add_node("classify_ticket", classify_ticket)
builder.add_node("draft_reply", draft_reply)
builder.add_edge(START, "classify_ticket")
builder.add_edge("classify_ticket", "draft_reply")
builder.add_edge("draft_reply", END)

basic_graph = builder.compile()

In [4]:
basic_result = basic_graph.invoke(
    {
        "ticket_id": "SUP-1042",
        "customer_message": (
            "I was charged twice for order 4821. Please refund the duplicate."
        ),
    }
)
basic_result

{'ticket_id': 'SUP-1042',
 'customer_message': 'I was charged twice for order 4821. Please refund the duplicate.',
 'category': 'billing',
 'summary': 'Customer reports a duplicate charge and requests a refund.',
 'draft_reply': 'Hi, I’m sorry about the duplicate charge on ticket SUP-1042. We’ll review the transaction and follow up.'}

`invoke` returns the final ticket state. The input provides the ticket ID and customer message; the graph fills in its category, summary, and draft reply.

Try drawing the graph with `basic_graph.get_graph().draw_mermaid()` if Mermaid rendering is available in your notebook environment.

In [5]:
print(basic_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_ticket(classify_ticket)
	draft_reply(draft_reply)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_ticket;
	classify_ticket --> draft_reply;
	draft_reply --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Branching by ticket category

A conditional edge calls a routing function after classification. Billing tickets go to a billing response node; account-access tickets go to an account support node.

In [6]:
class BranchState(TypedDict):
    customer_message: str
    category: str
    response: str


def classify_support_ticket(state: BranchState) -> dict:
    message = state["customer_message"].lower()
    if "password" in message or "locked" in message:
        category = "account_access"
    else:
        category = "billing"
    return {"category": category}


def respond_to_billing(state: BranchState) -> dict:
    return {
        "response": (
            "Billing support will check the transaction and contact you."
        )
    }


def respond_to_account_access(state: BranchState) -> dict:
    return {
        "response": (
            "Account support will help you restore access to your account."
        )
    }


def route_ticket(state: BranchState) -> str:
    if state["category"] == "account_access":
        return "account_support"
    return "billing_support"


branch_builder = StateGraph(BranchState)
branch_builder.add_node("classify", classify_support_ticket)
branch_builder.add_node("billing_support", respond_to_billing)
branch_builder.add_node("account_support", respond_to_account_access)
branch_builder.add_edge(START, "classify")
branch_builder.add_conditional_edges(
    "classify",
    route_ticket,
    {
        "billing_support": "billing_support",
        "account_support": "account_support",
    },
)
branch_builder.add_edge("billing_support", END)
branch_builder.add_edge("account_support", END)

branch_graph = branch_builder.compile()

In [7]:
branch_graph.invoke(
    {"customer_message": "I was charged twice for order 4821."}
)

{'customer_message': 'I was charged twice for order 4821.',
 'category': 'billing',
 'response': 'Billing support will check the transaction and contact you.'}

In [8]:
branch_graph.invoke(
    {"customer_message": "I am locked out and cannot reset my password."}
)

{'customer_message': 'I am locked out and cannot reset my password.',
 'category': 'account_access',
 'response': 'Account support will help you restore access to your account.'}

The routing function returns the destination node. Try changing the ticket text to see how the category selects a different support path. If one node must both update state and select the next node, LangGraph also supports returning a `Command`.

## 3. A bounded response-review loop

Before sending a ticket response, a workflow can check it against a simple support policy: acknowledge the issue and explain the next step. The loop revises an incomplete draft until it passes, with a maximum of three reviews.

In [9]:
class ReviewState(TypedDict):
    draft_reply: str
    review_count: int
    approved: bool


def review_reply(state: ReviewState) -> dict:
    review_count = state["review_count"] + 1
    meets_policy = "sorry" in state["draft_reply"].lower()
    approved = meets_policy or review_count >= 3
    return {"review_count": review_count, "approved": approved}


def route_after_review(state: ReviewState) -> str:
    if state["approved"]:
        return "finish"
    return "revise"


def revise_reply(state: ReviewState) -> dict:
    draft = (
        "I’m sorry about this issue. We’ll review the transaction and follow up. "
        f"{state['draft_reply']}"
    )
    return {"draft_reply": draft}


def finish_review(state: ReviewState) -> dict:
    return {"draft_reply": f"Ready to send: {state['draft_reply']}"}


loop_builder = StateGraph(ReviewState)
loop_builder.add_node("review", review_reply)
loop_builder.add_node("revise", revise_reply)
loop_builder.add_node("finish", finish_review)
loop_builder.add_edge(START, "review")
loop_builder.add_conditional_edges(
    "review",
    route_after_review,
    {"revise": "revise", "finish": "finish"},
)
loop_builder.add_edge("revise", "review")
loop_builder.add_edge("finish", END)

loop_graph = loop_builder.compile()

In [10]:
loop_graph.invoke(
    {
        "draft_reply": "We will check your order and follow up.",
        "review_count": 0,
        "approved": False,
    }
)

{'draft_reply': 'Ready to send: I’m sorry about this issue. We’ll review the transaction and follow up. We will check your order and follow up.',
 'review_count': 2,
 'approved': True}

The edge `revise → review` creates a cycle. Here the policy check or the three-review limit ends the loop. In production, keep explicit stop conditions and consider a graph recursion limit as a final safety bound.

## 4. Persistence: ticket checkpoints and thread IDs

A checkpointer saves graph state at execution steps. Use the ticket ID as a stable `thread_id` so the workflow can be inspected or continued later. `InMemorySaver` is useful for demonstrations, but its data disappears when the Python process exits. For durable deployments, use a database-backed checkpointer.

In [11]:
from langgraph.checkpoint.memory import InMemorySaver


class TicketProgressState(TypedDict):
    ticket_id: str
    review_count: int


def record_review(state: TicketProgressState) -> dict:
    return {"review_count": state["review_count"] + 1}


ticket_builder = StateGraph(TicketProgressState)
ticket_builder.add_node("record_review", record_review)
ticket_builder.add_edge(START, "record_review")
ticket_builder.add_edge("record_review", END)

ticket_saver = InMemorySaver()
ticket_graph = ticket_builder.compile(checkpointer=ticket_saver)
ticket_config = {"configurable": {"thread_id": "SUP-1042"}}

In [12]:
first_review = ticket_graph.invoke(
    {"ticket_id": "SUP-1042", "review_count": 0},
    ticket_config,
)
first_review

{'ticket_id': 'SUP-1042', 'review_count': 1}

In [13]:
# Continue this ticket's workflow using its saved checkpoint.
second_review = ticket_graph.invoke({}, ticket_config)
second_review

{'ticket_id': 'SUP-1042', 'review_count': 2}

In [14]:
# A different ticket has an independent thread and checkpoint.
other_ticket = {"configurable": {"thread_id": "SUP-2048"}}
ticket_graph.invoke(
    {"ticket_id": "SUP-2048", "review_count": 0},
    other_ticket,
)

{'ticket_id': 'SUP-2048', 'review_count': 1}

The checkpointer stores graph snapshots by thread, so ticket `SUP-1042` and ticket `SUP-2048` have separate histories. This is short-term workflow persistence, not a long-term customer profile store. For durable storage across process restarts, use a database-backed checkpointer; for customer facts shared across tickets, use a LangGraph store.

## 5. Conversation memory for a support thread

For chat history, the `add_messages` reducer appends new messages and handles message IDs. Combined with a checkpointer and a `thread_id`, it keeps the short-term conversation history for one support thread.

In [15]:
from typing import Annotated

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages


class ConversationState(TypedDict):
    messages: Annotated[list, add_messages]


def reply_in_support_thread(state: ConversationState) -> dict:
    latest_message = next(
        message
        for message in reversed(state["messages"])
        if isinstance(message, HumanMessage)
    )
    if "4821" in latest_message.content:
        response = "Thanks, I have order 4821. I’ll attach it to your billing ticket."
    else:
        response = "I’m sorry about the duplicate charge. What is your order number?"
    return {"messages": [AIMessage(content=response)]}


conversation_builder = StateGraph(ConversationState)
conversation_builder.add_node("support_reply", reply_in_support_thread)
conversation_builder.add_edge(START, "support_reply")
conversation_builder.add_edge("support_reply", END)

conversation_saver = InMemorySaver()
conversation_graph = conversation_builder.compile(
    checkpointer=conversation_saver
)
conversation_config = {
    "configurable": {"thread_id": "SUP-1042-chat"}
}

In [16]:
conversation_graph.invoke(
    {"messages": [HumanMessage(content="I was charged twice.")]},
    conversation_config,
)["messages"]

[HumanMessage(content='I was charged twice.', additional_kwargs={}, response_metadata={}, id='1f7f6b86-b215-47b2-ad71-e98e27431ecc'),
 AIMessage(content='I’m sorry about the duplicate charge. What is your order number?', additional_kwargs={}, response_metadata={}, id='2c4b21c2-a3f4-4647-a7b4-2a4a09fe1229', tool_calls=[], invalid_tool_calls=[])]

In [17]:
conversation_graph.invoke(
    {"messages": [HumanMessage(content="The order number is 4821.")]},
    conversation_config,
)["messages"]

[HumanMessage(content='I was charged twice.', additional_kwargs={}, response_metadata={}, id='1f7f6b86-b215-47b2-ad71-e98e27431ecc'),
 AIMessage(content='I’m sorry about the duplicate charge. What is your order number?', additional_kwargs={}, response_metadata={}, id='2c4b21c2-a3f4-4647-a7b4-2a4a09fe1229', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='The order number is 4821.', additional_kwargs={}, response_metadata={}, id='991526d8-3a43-4484-94fe-ea2e458e94a8'),
 AIMessage(content='Thanks, I have order 4821. I’ll attach it to your billing ticket.', additional_kwargs={}, response_metadata={}, id='e5fb7f35-e5a5-4ec3-b90a-769f68153f8c', tool_calls=[], invalid_tool_calls=[])]

The second turn uses the same thread, so the graph retains both the earlier question and the customer’s order number. A model-backed support node could use this accumulated history to draft the next response. A different `thread_id` starts a separate conversation.

## 6. Human approval before a refund

Refunds are consequential actions, so pause the ticket for a human decision before processing one. The graph needs a checkpointer, and you resume with the same `thread_id` using `Command(resume=...)`. The resume value becomes the return value from `interrupt()` inside the node.

In [18]:
from langgraph.types import Command, interrupt


class RefundState(TypedDict):
    ticket_id: str
    refund_amount: float
    status: str


def request_refund_approval(state: RefundState) -> dict:
    approved = interrupt(
        {
            "question": "Approve this refund?",
            "ticket_id": state["ticket_id"],
            "amount": state["refund_amount"],
        }
    )
    status = "approved_for_processing" if approved else "rejected"
    return {"status": status}


refund_builder = StateGraph(RefundState)
refund_builder.add_node("human_approval", request_refund_approval)
refund_builder.add_edge(START, "human_approval")
refund_builder.add_edge("human_approval", END)
refund_graph = refund_builder.compile(checkpointer=InMemorySaver())
refund_config = {
    "configurable": {"thread_id": "SUP-1042-refund"}
}

In [19]:
paused_result = refund_graph.invoke(
    {
        "ticket_id": "SUP-1042",
        "refund_amount": 24.99,
        "status": "pending_approval",
    },
    refund_config,
)
paused_result

{'ticket_id': 'SUP-1042',
 'refund_amount': 24.99,
 'status': 'pending_approval',
 '__interrupt__': [Interrupt(value={'question': 'Approve this refund?', 'ticket_id': 'SUP-1042', 'amount': 24.99}, id='17fed8c6317238cc396a8d4b58debb89', response_schema=None)]}

In [20]:
# The human approves; resume the paused ticket thread.
refund_graph.invoke(Command(resume=True), refund_config)

{'ticket_id': 'SUP-1042',
 'refund_amount': 24.99,
 'status': 'approved_for_processing'}

The initial result contains a `__interrupt__` entry with the ticket and proposed amount. In a real application, show that payload to an authorized reviewer and pass their decision back as the resume value. Keep the actual refund API call after approval.

## 7. Inspect and edit a saved ticket state

A reviewer may correct a ticket priority or update a draft before continuing. `get_state` inspects the latest checkpoint, and `update_state` applies the human’s correction.

In [21]:
edit_thread = {"configurable": {"thread_id": "SUP-1042-edit"}}
edit_builder = StateGraph(AgentState)
edit_builder.add_node("classify_ticket", classify_ticket)
edit_builder.add_node("draft_reply", draft_reply)
edit_builder.add_edge(START, "classify_ticket")
edit_builder.add_edge("classify_ticket", "draft_reply")
edit_builder.add_edge("draft_reply", END)
edit_graph = edit_builder.compile(checkpointer=InMemorySaver())
edit_graph.invoke(
    {
        "ticket_id": "SUP-1042",
        "customer_message": "I was charged twice for order 4821.",
    },
    edit_thread,
)

snapshot_before = edit_graph.get_state(edit_thread)
print(snapshot_before.values)

{'ticket_id': 'SUP-1042', 'customer_message': 'I was charged twice for order 4821.', 'category': 'billing', 'summary': 'Customer reports a duplicate charge and requests a refund.', 'draft_reply': 'Hi, I’m sorry about the duplicate charge on ticket SUP-1042. We’ll review the transaction and follow up.'}


In [22]:
edit_graph.update_state(
    edit_thread,
    {"draft_reply": "I confirmed the duplicate charge. A refund is pending."},
)
snapshot_after = edit_graph.get_state(edit_thread)
print(snapshot_after.values)

{'ticket_id': 'SUP-1042', 'customer_message': 'I was charged twice for order 4821.', 'category': 'billing', 'summary': 'Customer reports a duplicate charge and requests a refund.', 'draft_reply': 'I confirmed the duplicate charge. A refund is pending.'}


In [23]:
# The reviewer-edited reply is now in the saved ticket state.
edit_graph.get_state(edit_thread).values["draft_reply"]

'I confirmed the duplicate charge. A refund is pending.'

`update_state` creates a checkpoint with the requested changes. For message history, updates use the reducer; use message IDs when you mean to replace a particular message. Inspect the snapshot after editing so the correction is visible and auditable.

## Summary

- Model ticket data with a typed state and keep nodes focused on one task.
- Start with a straight-line DAG, then route billing and account tickets down different branches.
- Use a bounded review loop to check the response before sending.
- A checkpointer plus a ticket `thread_id` stores workflow state for follow-up.
- `add_messages` keeps the conversation history for a support thread.
- `interrupt` and `Command(resume=...)` pause for human refund approval.
- `get_state` and `update_state` let a reviewer inspect and correct saved ticket data.

### Further reading

- [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [Memory](https://docs.langchain.com/oss/python/langgraph/add-memory)
- [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)

## 8. Optional: draft the support reply with a model

This final example uses the model configuration loaded from `.env` at the top of the notebook. The ticket workflow still runs without credentials; this optional node calls the configured model to draft a concise, empathetic support reply.

In [24]:
class ModelState(TypedDict):
    ticket_id: str
    customer_message: str
    draft_reply: str


def draft_with_model(state: ModelState) -> dict:
    prompt = (
        "You are a customer support agent. Draft a concise, empathetic reply. "
        "Do not promise a refund; say the billing team will review the charge.\n"
        f"Ticket: {state['ticket_id']}\n"
        f"Customer message: {state['customer_message']}"
    )
    response = model.invoke(prompt)
    return {"draft_reply": response.content}


model_builder = StateGraph(ModelState)
model_builder.add_node("draft_reply", draft_with_model)
model_builder.add_edge(START, "draft_reply")
model_builder.add_edge("draft_reply", END)
model_graph = model_builder.compile()

if model is not None:
    model_result = model_graph.invoke(
        {
            "ticket_id": "SUP-1042",
            "customer_message": "I was charged twice for order 4821.",
        }
    )
    print(model_result["draft_reply"])
else:
    print("Model call skipped because .env is not configured.")

Hi, I’m sorry for the duplicate charge on order 4821. I’ve forwarded this to our billing team, and they’ll review the charge and follow up with you.


## 9. Tool calls: look up an order from a support ticket

A tool is a Python function the model can request. This graph gives the model an order lookup tool, routes tool calls to `ToolNode`, and sends the tool result back to the model. The model may then answer the customer or request another tool. This example uses a tiny in-memory order catalog; replace it with an authorized service/API in a real support system.

In [25]:
from typing import Annotated

from langchain_core.messages import (
    AIMessage,
    AnyMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


ORDER_STATUS = {
    "4821": "delivered on 2026-10-29",
    "7340": "in transit; expected delivery is 2026-11-04",
}


@tool
def lookup_order_status(order_id: str) -> str:
    """Look up the shipping status for a customer order ID."""
    normalized_id = order_id.strip()
    status = ORDER_STATUS.get(normalized_id)
    if status is None:
        return f"Order {normalized_id} was not found. Check the order ID."
    return f"Order {normalized_id} is {status}."


ticket_tools = [lookup_order_status]


class ToolCallState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


def call_support_model(state: ToolCallState) -> dict:
    response = support_model.invoke(state["messages"])
    return {"messages": [response]}


if model is not None:
    support_model = model.bind_tools(
        ticket_tools,
        parallel_tool_calls=False,
    )
    tool_builder = StateGraph(ToolCallState)
    tool_builder.add_node("assistant", call_support_model)
    tool_builder.add_node("tools", ToolNode(ticket_tools))
    tool_builder.add_edge(START, "assistant")
    tool_builder.add_conditional_edges("assistant", tools_condition)
    tool_builder.add_edge("tools", "assistant")
    support_tool_graph = tool_builder.compile()
    print("Tool-call graph is ready.")
else:
    print("Configure .env before building the model-backed tool-call graph.")

Tool-call graph is ready.


In [26]:
if model is not None:
    tool_result = support_tool_graph.invoke(
        {
            "messages": [
                SystemMessage(
                    content=(
                        "You are a support assistant. For order status questions, "
                        "call lookup_order_status before answering."
                    )
                ),
                HumanMessage(content="Where is my order 4821?"),
            ]
        }
    )
    for message in tool_result["messages"]:
        if isinstance(message, ToolMessage):
            print(f"Tool result: {message.content}")
        elif isinstance(message, AIMessage) and message.tool_calls:
            print(f"Tool request: {message.tool_calls}")
        elif isinstance(message, AIMessage):
            print(f"Assistant reply: {message.content}")

Tool request: [{'name': 'lookup_order_status', 'args': {'order_id': '4821'}, 'id': 'call_fHeYqvJxIWY7Owo4j5a3aOyY', 'type': 'tool_call'}]
Tool result: Order 4821 is delivered on 2026-10-29.
Assistant reply: Order **4821** was delivered on **October 29, 2026**.


Follow the execution path in the graph: `assistant → tools → assistant`. The first model response contains a structured tool call; `ToolNode` executes `lookup_order_status` and adds a `ToolMessage`; then the model receives that result and writes the final answer. `tools_condition` routes directly to `END` when the assistant response has no tool calls.

The tool only reads a local demo catalog. For a real refund or account change, validate the caller and ticket, and use the separate human approval pattern before performing the side effect.